Configuración

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import os

In [2]:
# Cargar el conjunto de datos completo desde el directorio local
try:
    df_completo = pd.read_csv('prosperLoanData.csv', sep=';')
    print("¡Éxito! El dataset original ha sido cargado por completo.")
except FileNotFoundError:
    print("Error: No se encontró el archivo 'prosperLoanData.csv' en la ruta especificada.")
 
# Mostrar las dimensiones del dataset original (filas, columnas)
print(f"Dimensiones del dataset original: {df_completo.shape[0]} filas y {df_completo.shape[1]} columnas.\n")
 
# Mostrar las primeras 5 filas para inspeccionar visualmente los datos
df_completo.head()

¡Éxito! El dataset original ha sido cargado por completo.
Dimensiones del dataset original: 113937 filas y 81 columnas.



,ListingKey,ListingNumber,ListingCreationDate,CreditGrade,Term,LoanStatus,ClosedDate,BorrowerAPR,BorrowerRate,LenderYield,...,LP_ServiceFees,LP_CollectionFees,LP_GrossPrincipalLoss,LP_NetPrincipalLoss,LP_NonPrincipalRecoverypayments,PercentFunded,Recommendations,InvestmentFromFriendsCount,InvestmentFromFriendsAmount,Investors
0,1021339766868145413AB3B,193129,09:29.3,C,36,Completed,14/08/2009 0:00,0.16516,0.1580,0.1380,...,-133.18,0.0,0.0,0.0,0.0,1.0,0,0,0.0,258
1,10273602499503308B223C1,1209647,28:07.9,NaN,36,Current,NaN,0.12016,0.0920,0.0820,...,0.00,0.0,0.0,0.0,0.0,1.0,0,0,0.0,1
2,0EE9337825851032864889A,81716,00:47.1,HR,36,Completed,17/12/2009 0:00,0.28269,0.2750,0.2400,...,-24.20,0.0,0.0,0.0,0.0,1.0,0,0,0.0,41
3,0EF5356002482715299901A,658116,02:35.0,NaN,36,Current,NaN,0.12528,0.0974,0.0874,...,-108.01,0.0,0.0,0.0,0.0,1.0,0,0,0.0,158
4,0F023589499656230C5E3E2,909464,38:39.1,NaN,36,Current,NaN,0.24614,0.2085,0.1985,...,-60.27,0.0,0.0,0.0,0.0,1.0,0,0,0.0,20


## Limpieza inicial de datos

Se procede a realizar la limpieza inicial de loa datos, identificar y eliminar duplicados, eliminar registros sin datos de geolocalización.

In [3]:
# Eliminación de duplicados
num_duplicados = df_completo.duplicated().sum()
if num_duplicados > 0:
    df_completo.drop_duplicates(keep='first', inplace=True)
    print(f"Limpieza: Se eliminaron {num_duplicados} filas duplicadas.")

# Validación de filas y columnas
print(f"Dimensiones del dataset original: {df_completo.shape[0]} filas y {df_completo.shape[1]} columnas.\n")
 

Dimensiones del dataset original: 113937 filas y 81 columnas.



La data no cuenta con ducplicados y se mantienen las filas originales, sin embargo, dadas las limitantes de procesamiento y el objetivo específico se realizó una clasificación de las variables siginificativas que aportarán al entrenamiento y desarrollo del modelo, por lo cual se decide conservar 11 de las 81 columnas iniciales

In [4]:
# Definir la lista de las 11 variables clave para el análisis financiero-climático
columnas_relevantes = [
    'BorrowerState', 'LoanStatus', 'EstimatedEffectiveYield',
    'BorrowerAPR', 'DebtToIncomeRatio', 'IncomeRange',
    'StatedMonthlyIncome', 'Term', 'LoanOriginalAmount',
    'MonthlyLoanPayment', 'Recommendations'
]
 
# Crear el nuevo DataFrame conservando solo las columnas seleccionadas y eliminando clientes sin
# geolocalización
df_ecoscore = df_completo.dropna(subset=['BorrowerState']).reset_index(drop=True)[columnas_relevantes].copy()


In [9]:
# Mostrar la nueva dimensión del dataset optimizado
print("--- Dataset Optimizado para EcoScore ---")
print(f"Dimensiones finales: {df_ecoscore.shape[0]} filas y {df_ecoscore.shape[1]} columnas.")
df_ecoscore.head()

--- Dataset Optimizado para EcoScore ---
Dimensiones finales: 108422 filas y 16 columnas.


,BorrowerState,LoanStatus,EstimatedEffectiveYield,BorrowerAPR,DebtToIncomeRatio,IncomeRange,StatedMonthlyIncome,Term,LoanOriginalAmount,MonthlyLoanPayment,Recommendations,Precipitation_Risk_Index,Financial_Status,Target_Risk_Scale,IncomeRange_Encoded,BorrowerState_Encoded
0,CO,Completed,NaN,0.16516,0.17,"$25,000-49,999",3083.333333,36,9425,330.43,0,50.0,Saludable,2,2.0,0.020383
1,CO,Current,0.07960,0.12016,0.18,"$50,000-74,999",6125.000000,36,10000,318.93,0,50.0,Saludable,2,3.0,0.020383
2,GA,Completed,NaN,0.28269,0.06,Not displayed,2083.333333,36,3001,123.32,0,70.4,Saludable,2,0.0,0.046190
3,GA,Current,0.08490,0.12528,0.15,"$25,000-49,999",2875.000000,36,10000,321.45,0,70.4,Saludable,2,2.0,0.046190
4,MN,Current,0.18316,0.24614,0.26,"$100,000+",9583.333333,36,15000,563.97,0,50.0,Saludable,2,5.0,0.021379


# Cálculo de Riesgo por Precipitación Extrema

Para saber el riesgo por precipitación extrema se creará variable `climate_risk_matrix` aplicando matriz de vulnerabilidad hídrica histórica por Estado, dado que es un enfoque académico los valores asignados al Índice de Riesgo por Precipitación Extrema en el modelo EcoScore fueron normalizados en una escala de 0 a 100, tomando como referencia los mapas de riesgo histórico de inundaciones de la Agencia Federal para la Gestión de Emergencias (FEMA) y los registros institucionales de eventos meteorológicos severos de la Administración Nacional Oceánica y Atmosférica (NOAA) de los Estados Unidos.

In [10]:
# Matriz de vulnerabilidad hídrica histórica por Estado (Escala 0 a 100)
climate_risk_matrix = {
    'FL': 94.2, 'LA': 91.0, 'TX': 83.5, 'NC': 77.1, 'GA': 70.4,
    'NY': 56.8, 'CA': 48.0, 'OH': 40.2, 'NV': 14.1, 'UT': 10.5
}
df_ecoscore['Precipitation_Risk_Index'] = df_ecoscore['BorrowerState'].map(climate_risk_matrix).fillna(50.0)

df_ecoscore.head(5)


,BorrowerState,LoanStatus,EstimatedEffectiveYield,BorrowerAPR,DebtToIncomeRatio,IncomeRange,StatedMonthlyIncome,Term,LoanOriginalAmount,MonthlyLoanPayment,Recommendations,Precipitation_Risk_Index,Financial_Status,Target_Risk_Scale,IncomeRange_Encoded,BorrowerState_Encoded
0,CO,Completed,NaN,0.16516,0.17,"$25,000-49,999",3083.333333,36,9425,330.43,0,50.0,Saludable,2,2.0,0.020383
1,CO,Current,0.07960,0.12016,0.18,"$50,000-74,999",6125.000000,36,10000,318.93,0,50.0,Saludable,2,3.0,0.020383
2,GA,Completed,NaN,0.28269,0.06,Not displayed,2083.333333,36,3001,123.32,0,70.4,Saludable,2,0.0,0.046190
3,GA,Current,0.08490,0.12528,0.15,"$25,000-49,999",2875.000000,36,10000,321.45,0,70.4,Saludable,2,2.0,0.046190
4,MN,Current,0.18316,0.24614,0.26,"$100,000+",9583.333333,36,15000,563.97,0,50.0,Saludable,2,5.0,0.021379


# Preparación de dataset para el modelo


En este paso se crearán la variable objetivo en rangos de escala, mapeo y conversión de variables categóricas, se tratan los nulos y se hace la selección final de features para el modelo

In [11]:
# =====================================================================
# 1. PASO 1: CREACIÓN DE LA VARIABLE OBJETIVO (RANGOS DE ESCALA)
# =====================================================================

# Clasificar el estado financiero real del crédito
def clasificar_financiero(status):
    if status in ['Completed', 'Current']:
        return 'Saludable'
    elif 'Past Due' in str(status):
        return 'Alerta'
    elif status in ['Defaulted', 'Charged-Off']:
        return 'Impago'
    else:
        return 'Saludable'  # Criterio conservador para estados intermedios

df_ecoscore['Financial_Status'] = df_ecoscore['LoanStatus'].apply(clasificar_financiero)

# Generar la matriz híbrida de 4 escalas (Financiera + EcoScore)
def asignar_escala_riesgo(row):
    fin = row['Financial_Status']
    clima = row['Precipitation_Risk_Index']
    
    # Escala 4: Crítico (Impago definitivo y/o zona de desastre inminente)
    if fin == 'Impago' or (clima > 75.0 and fin == 'Alerta'):
        return 4
    # Escala 3: Alto (Alertas de pago o zona de alta inundación con finanzas sanas)
    elif fin == 'Alerta' or (clima > 75.0 and fin == 'Saludable') or (clima > 50.0 and fin == 'Alerta'):
        return 3
    # Escala 2: Moderado (Zonas de riesgo medio con finanzas estables)
    elif clima > 40.0:
        return 2
    # Escala 1: Bajo (Finanzas sanas en zonas con bajo índice de inundación)
    else:
        return 1

df_ecoscore['Target_Risk_Scale'] = df_ecoscore.apply(asignar_escala_riesgo, axis=1)

# =====================================================================
# 2. PASO 2: CONVERSIÓN Y MAPEO DE VARIABLES CATEGÓRICAS
# =====================================================================

# Mapeo manual y ordenado de los rangos de ingresos (Variable Ordinal)
mapa_ingresos = {
    'Not displayed': 0, 'Not Employed': 0, '$0': 0,
    '$1-24,999': 1, '$25,000-49,999': 2, 
    '$50,000-74,999': 3, '$75,000-99,999': 4, 
    '$100,000+': 5
}
df_ecoscore['IncomeRange_Encoded'] = df_ecoscore['IncomeRange'].map(mapa_ingresos).fillna(0)

# Codificación por frecuencias para BorrowerState (evita expandir la dimensionalidad)
state_freq = df_ecoscore['BorrowerState'].value_counts(normalize=True)
df_ecoscore['BorrowerState_Encoded'] = df_ecoscore['BorrowerState'].map(state_freq)

# =====================================================================
# 3. PASO 3: TRATAMIENTO DE NULOS E INGENIERÍA DE CARACTERÍSTICAS
# =====================================================================

# Imputar nulos en la relación deuda/ingresos con la mediana
mediana_debt = df_ecoscore['DebtToIncomeRatio'].median()
df_ecoscore['DebtToIncomeRatio'].fillna(mediana_debt, inplace=True)

# Imputar nulos en el rendimiento efectivo si los hubiera
df_ecoscore['EstimatedEffectiveYield'].fillna(df_ecoscore['EstimatedEffectiveYield'].median(), inplace=True)

# =====================================================================
# 4. PASO 4: SELECCIÓN FINAL DE FEATURES PARA EL MODELO
# =====================================================================

features_modelo = [
    'Precipitation_Risk_Index', 'EstimatedEffectiveYield', 'BorrowerAPR', 
    'DebtToIncomeRatio', 'IncomeRange_Encoded', 'StatedMonthlyIncome', 
    'Term', 'LoanOriginalAmount', 'MonthlyLoanPayment', 
    'Recommendations', 'BorrowerState_Encoded'
]

X = df_ecoscore[features_modelo]
y = df_ecoscore['Target_Risk_Scale']

print("--- Distribución de las Escalas de Riesgo Creadas ---")
print(y.value_counts().sort_index())
print(f"\nDataset listo para entrenamiento. X: {X.shape}, y: {y.shape}")


--- Distribución de las Escalas de Riesgo Creadas ---
Target_Risk_Scale
1     1834
2    83508
3    18302
4     4778
Name: count, dtype: int64

Dataset listo para entrenamiento. X: (108422, 11), y: (108422,)


C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_6236\4286842578.py:61: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/docs/dev/user_guide/migration.html#copy-on-write-cow
  df_ecoscore['DebtToIncomeRatio'].fillna(mediana_debt, inplace=True)
C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_6236\4286842578.py:64: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series t

In [13]:
# Guardar el DataFrame optimizado en un archivo CSV
df_ecoscore.to_csv('df_ecoscore.csv', index=False)
print(f"Dimensiones finales: {df_ecoscore.shape[0]} filas y {df_ecoscore.shape[1]} columnas.")

# Se realiza una última validación de la vista de los datos
df_ecoscore.head(5)

Dimensiones finales: 108422 filas y 16 columnas.


,BorrowerState,LoanStatus,EstimatedEffectiveYield,BorrowerAPR,DebtToIncomeRatio,IncomeRange,StatedMonthlyIncome,Term,LoanOriginalAmount,MonthlyLoanPayment,Recommendations,Precipitation_Risk_Index,Financial_Status,Target_Risk_Scale,IncomeRange_Encoded,BorrowerState_Encoded
0,CO,Completed,NaN,0.16516,0.17,"$25,000-49,999",3083.333333,36,9425,330.43,0,50.0,Saludable,2,2.0,0.020383
1,CO,Current,0.07960,0.12016,0.18,"$50,000-74,999",6125.000000,36,10000,318.93,0,50.0,Saludable,2,3.0,0.020383
2,GA,Completed,NaN,0.28269,0.06,Not displayed,2083.333333,36,3001,123.32,0,70.4,Saludable,2,0.0,0.046190
3,GA,Current,0.08490,0.12528,0.15,"$25,000-49,999",2875.000000,36,10000,321.45,0,70.4,Saludable,2,2.0,0.046190
4,MN,Current,0.18316,0.24614,0.26,"$100,000+",9583.333333,36,15000,563.97,0,50.0,Saludable,2,5.0,0.021379
